Install mlflow and pyngrok

In [1]:
!pip install mlflow --quiet
!pip install pyngrok --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.8/538.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.9/202.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0

Run MLflow tracking UI in the background

In [2]:
get_ipython().system_raw("mlflow ui --port 5000 &") # run tracking UI in the background

Run ngrok to provide external access to the UI.

In [3]:
from pyngrok import ngrok
from getpass import getpass

# Terminate open tunnels if exist
ngrok.kill()

# Get your authtoken from https://dashboard.ngrok.com/auth
NGROK_AUTH_TOKEN = getpass("Enter your ngrok auth token (create one at https://ngrok.com/): ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open an HTTPs tunnel on port 5000 for http://localhost:5000
ngrok_tunnel = ngrok.connect(addr="5000", proto="http", bind_tls=True)
print("MLflow Tracking UI:", ngrok_tunnel.public_url)

Enter your ngrok auth token (create one at https://ngrok.com/): ··········
MLflow Tracking UI: https://fb41-34-75-226-240.ngrok-free.app


Install additional dependencies

In [4]:
!pip install scikit-learn
!pip install matplotlib
!pip install pandas

Run model training and save the results to mlflow

In [5]:
import mlflow
from sklearn import datasets, model_selection, linear_model, metrics
import matplotlib.pyplot as plt
import pandas as pd
import pickle

mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Load the dataset
data = datasets.load_diabetes(as_frame=True)
X = data.data
y = data.target

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the model
model = linear_model.LinearRegression()

# Start an MLflow run
with mlflow.start_run(run_name="example_run"):
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Calculate metrics
    mse = metrics.mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = metrics.r2_score(y_test, predictions)

    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")

    # Log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Plot and log residuals as an artifact
    residuals = y_test - predictions

    # Save and log the model as an artifact
    with open("linear_regression_model.pkl", "wb") as model_file:
        pickle.dump(model, model_file)
    mlflow.log_artifact("linear_regression_model.pkl")

    # Log a CSV of predictions
    results_df = pd.DataFrame({"Actual": y_test, "Predicted": predictions, "Residuals": residuals})
    results_df.to_csv("predictions.csv", index=False)
    mlflow.log_artifact("predictions.csv")

    print("Run completed successfully. Metrics and artifacts have been logged.")

2024/08/19 07:46:32 INFO mlflow.tracking._tracking_service.client: 🏃 View run example_run at: http://127.0.0.1:5000/#/experiments/0/runs/5800f4e90f9c4c38ae0aaa4d483f2d17.
2024/08/19 07:46:32 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0.


Run completed successfully. Metrics and artifacts have been logged.
